# Instance-Aware Person Removal: Single-Shot vs Progressive Inpainting
## CSE-498R Directed Research — Full Pipeline (200 COCO Images)

---

### পুরো Pipeline:
```
Local ZIP upload (200 COCO images)
        ↓
YOLOv8x-seg  →  Person Detection + Segmentation Mask
        ↓
Method A: Union Mask → LaMa Inpainting (সব একবারে)
Method B: Per-Person → LaMa Inpainting (ধাপে ধাপে, largest-first)
        ↓
PSNR / SSIM / LPIPS Metrics
        ↓
CSV checkpoint + Visual comparisons + Aggregate charts
        ↓
ZIP করে download
```

### মডেল:
| Model | কাজ | Size |
|---|---|---|
| **YOLOv8x-seg** | Person detection + pixel mask | ~137 MB |
| **LaMa** | Fast inpainting (Samsung Research) | ~200 MB |
| **LPIPS (AlexNet)** | Perceptual quality metric | ~9 MB |

---
⚠️ **Runtime → T4 GPU** চালু করো আগে: `Runtime → Change runtime type → T4 GPU`

## Cell 1 — Install Dependencies
⚠️ এই cell চালানোর পর **Runtime → Restart session** করো, তারপর Cell 2 থেকে শুরু করো।

In [ ]:
!pip install -q "numpy==1.26.4"
!pip install -q "scipy==1.13.1" "scikit-image==0.24.0"
!pip install -q ultralytics simple-lama-inpainting lpips opencv-python-headless

print('='*50)
print('All packages installed!')
print('='*50)
print()
print('IMPORTANT: Runtime → Restart session করো')
print('তারপর Cell 2 থেকে চালু করো।')

## Cell 2 — Upload Dataset ZIP
তোমার local `coco_200_persons.zip` ফাইলটা এখানে upload করো।

In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path

EXTRACT_DIR = '/content/coco_200_persons'

existing = list(Path(EXTRACT_DIR).glob('*.jpg')) if os.path.exists(EXTRACT_DIR) else []

if len(existing) >= 10:
    print(f'Dataset already extracted: {len(existing)} images found.')
    print('Skipping upload — continue to Cell 3.')
else:
    print('coco_200_persons.zip ফাইলটা select করো...')
    uploaded = files.upload()

    zip_name = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_name}  ({os.path.getsize(zip_name)/1e6:.1f} MB)')

    print('Extracting...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(EXTRACT_DIR)

    os.remove(zip_name)

    imgs = list(Path(EXTRACT_DIR).glob('*.jpg'))
    print(f'Extracted {len(imgs)} images to {EXTRACT_DIR}')

## Cell 3 — Imports & GPU Check

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import warnings, os, time, csv, traceback, random, shutil
from collections import Counter
warnings.filterwarnings('ignore')

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
import lpips
from ultralytics import YOLO
from simple_lama_inpainting import SimpleLama

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}')
    print(f'VRAM   : {props.total_memory/1e9:.1f} GB')
else:
    print('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

print('Cell 3 done!')

## Cell 4 — Load Models

In [ ]:
%%time
print('Loading YOLOv8x-seg (~137 MB)...')
yolo_seg = YOLO('yolov8x-seg.pt')
print('  YOLOv8x-seg done')

print('Loading LaMa inpainting (~200 MB)...')
lama = SimpleLama()
print('  LaMa done')

print('Loading LPIPS (AlexNet)...')
lpips_fn = lpips.LPIPS(net='alex').to(DEVICE)
print('  LPIPS done')

print('All models ready!')

## Cell 5 — Configuration

In [ ]:
IMAGE_DIR  = '/content/coco_200_persons'
SAVE_DIR   = '/content/results'
VISUAL_DIR = f'{SAVE_DIR}/visuals'
CSV_PATH   = f'{SAVE_DIR}/metrics.csv'

YOLO_CONF          = 0.40
MAX_PERSONS        = 8
MIN_MASK_AREA      = 2000
SAVE_VISUALS_N     = 30
RANDOM_VISUAL_SEED = 42
MASK_DILATE_KERNEL = 15
MASK_BLUR_RADIUS   = 21
IMG_EXTS           = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(VISUAL_DIR, exist_ok=True)

all_images = sorted([
    p for p in Path(IMAGE_DIR).rglob('*')
    if p.suffix.lower() in IMG_EXTS
])
print(f'Images found : {len(all_images)}')
if len(all_images) == 0:
    print('No images found! Make sure Cell 2 ran successfully.')
else:
    print(f'Sample      : {[p.name for p in all_images[:3]]}')

rng        = random.Random(RANDOM_VISUAL_SEED)
visual_set = set(map(str, rng.sample(all_images, min(SAVE_VISUALS_N, len(all_images)))))
print(f'Visual comparisons will save for {len(visual_set)} images')

CSV_FIELDS = [
    'image','n_persons',
    'A_PSNR','A_SSIM','A_LPIPS','A_mask_pct','A_time_s',
    'B_PSNR','B_SSIM','B_LPIPS','B_mask_pct','B_time_s',
    'status'
]
already_done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for row in csv.DictReader(f):
            if row.get('status') == 'ok':
                already_done.add(row['image'])
    print(f'Resume: {len(already_done)} done, {len(all_images)-len(already_done)} remaining')
else:
    with open(CSV_PATH, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()
    print('New CSV created.')

print('Cell 5 done!')

## Cell 6 — Core Pipeline Functions

In [ ]:
def detect_and_segment(image_bgr):
    H, W    = image_bgr.shape[:2]
    results = yolo_seg(image_bgr, conf=YOLO_CONF, classes=[0], verbose=False)[0]
    persons = []
    if results.masks is None or results.boxes is None:
        return persons
    masks_data = results.masks.data.cpu().numpy()
    boxes_data = results.boxes.xyxy.cpu().numpy()
    for i in range(len(boxes_data)):
        mask_full = cv2.resize(masks_data[i], (W, H), interpolation=cv2.INTER_LINEAR)
        binary    = (mask_full > 0.5).astype(np.uint8) * 255
        area      = int(binary.sum() / 255)
        if area < MIN_MASK_AREA:
            continue
        x1, y1, x2, y2 = map(int, boxes_data[i])
        persons.append({'bbox': [x1,y1,x2,y2], 'mask': binary, 'area': area})
    persons.sort(key=lambda p: p['area'], reverse=True)
    return persons[:MAX_PERSONS]


def dilate_and_feather(mask):
    kernel    = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (MASK_DILATE_KERNEL, MASK_DILATE_KERNEL))
    dilated   = cv2.dilate(mask, kernel, iterations=1)
    feathered = cv2.GaussianBlur(
        dilated.astype(np.float32), (MASK_BLUR_RADIUS, MASK_BLUR_RADIUS), 0)
    return (feathered > 127).astype(np.uint8) * 255


def lama_inpaint(image_rgb, mask):
    pil_img  = Image.fromarray(image_rgb)
    pil_mask = Image.fromarray(mask)
    result   = lama(pil_img, pil_mask)
    out_np   = np.array(result.convert('RGB'))
    mask_3   = np.stack([mask / 255.0] * 3, axis=-1)
    return (out_np * mask_3 + image_rgb * (1 - mask_3)).astype(np.uint8)


def method_single_shot(image_rgb, persons):
    """Method A: merge all masks -> single inpainting call"""
    H, W       = image_rgb.shape[:2]
    union_mask = np.zeros((H, W), dtype=np.uint8)
    for p in persons:
        union_mask = np.maximum(union_mask, dilate_and_feather(p['mask']))
    t0     = time.time()
    result = lama_inpaint(image_rgb, union_mask)
    return {'result': result, 'union_mask': union_mask,
            'time_s': round(time.time()-t0, 2)}


def method_progressive(image_rgb, persons):
    """Method B: remove one-by-one largest-first, each result feeds next step"""
    H, W     = image_rgb.shape[:2]
    current  = image_rgb.copy()
    cum_mask = np.zeros((H, W), dtype=np.uint8)
    steps    = []
    t0       = time.time()
    for p in persons:
        step_mask = dilate_and_feather(p['mask'])
        cum_mask  = np.maximum(cum_mask, step_mask)
        current   = lama_inpaint(current, step_mask)
        steps.append(current.copy())
    return {'result': current, 'union_mask': cum_mask,
            'step_images': steps, 'time_s': round(time.time()-t0, 2)}


def compute_metrics(original, reconstructed, mask):
    m = mask > 127
    if not m.any():
        return {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 1.0, 'mask_coverage_%': 0.0}
    ys, xs  = np.where(m)
    y1, y2  = ys.min(), ys.max()
    x1, x2  = xs.min(), xs.max()
    orig_c  = original[y1:y2, x1:x2]
    recon_c = reconstructed[y1:y2, x1:x2]
    if orig_c.shape[0] < 8 or orig_c.shape[1] < 8:
        return {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 1.0, 'mask_coverage_%': 0.0}
    psnr_val = psnr_fn(orig_c, recon_c, data_range=255)
    ssim_val = ssim_fn(orig_c, recon_c, channel_axis=2, data_range=255)
    def to_t(img):
        return (torch.from_numpy(img.astype(np.float32)/127.5-1.0)
                .permute(2,0,1).unsqueeze(0).to(DEVICE))
    with torch.no_grad():
        lp = lpips_fn(to_t(orig_c), to_t(recon_c)).item()
    return {
        'PSNR':            round(psnr_val, 3),
        'SSIM':            round(ssim_val, 4),
        'LPIPS':           round(lp, 4),
        'mask_coverage_%': round(100.0 * m.sum() / m.size, 2),
    }


def save_visual(stem, original, persons, res_A, res_B, mA, mB):
    seg_vis = original.copy().astype(np.float32)
    for p in persons:
        green        = np.zeros_like(original, dtype=np.float32)
        green[:,:,1] = p['mask'].astype(np.float32)
        alpha        = (p['mask']/255.0)[:,:,None]
        seg_vis      = seg_vis*(1-0.45*alpha) + green*0.45*alpha
    seg_vis = seg_vis.clip(0,255).astype(np.uint8)

    det_vis = original.copy()
    for i, p in enumerate(persons):
        x1,y1,x2,y2 = p['bbox']
        cv2.rectangle(det_vis,(x1,y1),(x2,y2),(0,220,80),3)
        cv2.putText(det_vis,f'P{i+1}',(x1+4,y1+36),
                    cv2.FONT_HERSHEY_SIMPLEX,1.0,(0,220,80),2)

    fig, axes = plt.subplots(1, 5, figsize=(28, 6))
    fig.suptitle(f'{stem}  |  {len(persons)} person(s)', fontsize=12, fontweight='bold')
    def show(ax, img, title, sub=''):
        ax.imshow(img); ax.set_title(f'{title}\n{sub}', fontsize=8); ax.axis('off')
    show(axes[0], original,        'Original')
    show(axes[1], det_vis,         'YOLO Detection',    f'{len(persons)} person(s)')
    show(axes[2], seg_vis,         'Segmentation Mask', 'YOLOv8x-seg')
    show(axes[3], res_A['result'], 'Method A — Single-Shot',
         f"PSNR={mA['PSNR']} SSIM={mA['SSIM']} LPIPS={mA['LPIPS']}\n"
         f"mask={mA['mask_coverage_%']}% | {res_A['time_s']}s")
    show(axes[4], res_B['result'], 'Method B — Progressive',
         f"PSNR={mB['PSNR']} SSIM={mB['SSIM']} LPIPS={mB['LPIPS']}\n"
         f"mask={mB['mask_coverage_%']}% | {res_B['time_s']}s")
    plt.tight_layout()
    plt.savefig(f'{VISUAL_DIR}/{stem}_comparison.jpg', dpi=110, bbox_inches='tight')
    plt.close(fig)

    steps = res_B.get('step_images', [])
    if len(steps) >= 2:
        n_cols = len(steps) + 1
        fig2, axes2 = plt.subplots(1, n_cols, figsize=(6*n_cols, 5))
        fig2.suptitle('Method B — Progressive Steps (largest-first)',
                      fontsize=11, fontweight='bold')
        axes2[0].imshow(original); axes2[0].set_title('Original'); axes2[0].axis('off')
        for j, s in enumerate(steps):
            axes2[j+1].imshow(s)
            axes2[j+1].set_title(f'After P{j+1} removed')
            axes2[j+1].axis('off')
        plt.tight_layout()
        plt.savefig(f'{VISUAL_DIR}/{stem}_progressive.jpg', dpi=100, bbox_inches='tight')
        plt.close(fig2)


def append_csv(row_dict):
    with open(CSV_PATH, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row_dict)


print('Cell 6 done — all functions defined!')

## Cell 7 — Main Batch Loop
এই cell চালালেই সব ২০০ ছবি automatically process হবে।  
Session disconnect হলে **Cell 3 → Cell 7** আবার চালাও — already done ছবি skip হবে।

In [ ]:
total             = len(all_images)
skipped_no_person = 0
errors            = []
t_global          = time.time()

print(f'Total images      : {total}')
print(f'Already processed : {len(already_done)}')
print(f'To process now    : {total - len(already_done)}')
print('='*68)

for idx, img_path in enumerate(all_images, 1):
    name = img_path.name
    stem = img_path.stem

    if name in already_done:
        print(f'[{idx:>3}/{total}] SKIP: {name}')
        continue

    print(f'\n[{idx:>3}/{total}] {name}')
    t_img = time.time()

    try:
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise ValueError('cv2.imread returned None')
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        H, W = image_rgb.shape[:2]
        print(f'  Size: {W}x{H}')

        persons = detect_and_segment(image_bgr)
        n = len(persons)
        print(f'  Persons: {n}')

        if n == 0:
            append_csv({'image':name,'n_persons':0,
                'A_PSNR':'-','A_SSIM':'-','A_LPIPS':'-','A_mask_pct':'-','A_time_s':'-',
                'B_PSNR':'-','B_SSIM':'-','B_LPIPS':'-','B_mask_pct':'-','B_time_s':'-',
                'status':'no_person'})
            skipped_no_person += 1
            print('  No person detected — skipped')
            continue

        print('  [A] Single-Shot...', end=' ', flush=True)
        res_A = method_single_shot(image_rgb, persons)
        mA    = compute_metrics(image_rgb, res_A['result'], res_A['union_mask'])
        print(f"PSNR={mA['PSNR']} SSIM={mA['SSIM']} LPIPS={mA['LPIPS']} | {res_A['time_s']}s")

        print(f'  [B] Progressive ({n} steps)...', end=' ', flush=True)
        res_B = method_progressive(image_rgb, persons)
        mB    = compute_metrics(image_rgb, res_B['result'], res_B['union_mask'])
        print(f"PSNR={mB['PSNR']} SSIM={mB['SSIM']} LPIPS={mB['LPIPS']} | {res_B['time_s']}s")

        if str(img_path) in visual_set:
            save_visual(stem, image_rgb, persons, res_A, res_B, mA, mB)
            print('  Visual saved')

        append_csv({
            'image':name,'n_persons':n,
            'A_PSNR':mA['PSNR'],  'A_SSIM':mA['SSIM'],
            'A_LPIPS':mA['LPIPS'],'A_mask_pct':mA['mask_coverage_%'],
            'A_time_s':res_A['time_s'],
            'B_PSNR':mB['PSNR'],  'B_SSIM':mB['SSIM'],
            'B_LPIPS':mB['LPIPS'],'B_mask_pct':mB['mask_coverage_%'],
            'B_time_s':res_B['time_s'],
            'status':'ok'
        })

        elapsed = time.time() - t_img
        done_n  = idx - len(already_done) - skipped_no_person - len(errors)
        eta_min = (time.time()-t_global) / max(done_n,1) * (total-idx) / 60
        print(f'  Done {elapsed:.1f}s | ETA ~ {eta_min:.0f} min')

        del res_A, res_B
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'  ERROR: {e}')
        traceback.print_exc()
        errors.append({'image': name, 'error': str(e)})
        append_csv({'image':name,'n_persons':-1,
            'A_PSNR':'-','A_SSIM':'-','A_LPIPS':'-','A_mask_pct':'-','A_time_s':'-',
            'B_PSNR':'-','B_SSIM':'-','B_LPIPS':'-','B_mask_pct':'-','B_time_s':'-',
            'status':'error'})
        torch.cuda.empty_cache()

total_min = (time.time() - t_global) / 60
print('\n' + '='*68)
print(f'BATCH COMPLETE  ({total_min:.1f} min total)')
print(f'  Processed : {total - len(already_done) - skipped_no_person - len(errors)}')
print(f'  No person : {skipped_no_person}')
print(f'  Errors    : {len(errors)}')
print(f'  CSV       : {CSV_PATH}')
print('='*68)

## Cell 8 — Aggregate Summary Table

In [ ]:
rows = []
with open(CSV_PATH) as f:
    for row in csv.DictReader(f):
        if row['status'] == 'ok':
            rows.append(row)

print(f'Valid results: {len(rows)} images\n')

def col(key): return np.array([float(r[key]) for r in rows])

A_psnr  = col('A_PSNR');  B_psnr  = col('B_PSNR')
A_ssim  = col('A_SSIM');  B_ssim  = col('B_SSIM')
A_lpips = col('A_LPIPS'); B_lpips = col('B_LPIPS')
A_time  = col('A_time_s');B_time  = col('B_time_s')
n_pers  = col('n_persons')

print('='*74)
print('  AGGREGATE RESULTS')
print('='*74)
print(f'{"Metric":<22} {"Method A (Single-Shot)":>23} {"Method B (Progressive)":>23}  Winner')
print('-'*74)
for label, a, b, direction in [
    ('PSNR  (higher=better)', A_psnr,  B_psnr,  'high'),
    ('SSIM  (higher=better)', A_ssim,  B_ssim,  'high'),
    ('LPIPS (lower=better)',  A_lpips, B_lpips, 'low'),
]:
    am, bm   = a.mean(), b.mean()
    as_, bs_ = a.std(),  b.std()
    winner   = ('B-Progressive' if (bm>am if direction=='high' else bm<am)
                else 'A-SingleShot')
    print(f'{label:<22} {am:>9.4f} +/- {as_:<8.4f} {bm:>9.4f} +/- {bs_:<8.4f}  {winner}')
print('-'*74)
print(f'{"Avg time/image (s)":<22} {A_time.mean():>23.2f} {B_time.mean():>23.2f}')
print(f'{"Total time (min)":<22} {A_time.sum()/60:>23.1f} {B_time.sum()/60:>23.1f}')
print('='*74)
print(f'N images: {len(rows)} | Avg persons/img: {n_pers.mean():.2f} | Max: {int(n_pers.max())} | Min: {int(n_pers.min())}')
print('='*74)

## Cell 9 — Figure 1: Box Plot

In [ ]:
CA, CB = '#e07070', '#5b9bd5'

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(f'Method A vs Method B | N={len(rows)} images', fontsize=13, fontweight='bold')
for ax, (a_arr, b_arr, title) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR (dB) - higher is better'),
    (A_ssim,  B_ssim,  'SSIM - higher is better'),
    (A_lpips, B_lpips, 'LPIPS - lower is better'),
]):
    bp = ax.boxplot([a_arr, b_arr], labels=['A: Single-Shot','B: Progressive'],
                    patch_artist=True, notch=True,
                    medianprops=dict(color='black', linewidth=2.5))
    for patch, c in zip(bp['boxes'], [CA, CB]):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for i, arr in enumerate([a_arr, b_arr], 1):
        ax.text(i, arr.max()*1.01, f'mean={arr.mean():.3f}',
                ha='center', fontsize=10, fontweight='bold', color=[CA,CB][i-1])
plt.tight_layout()
p = f'{SAVE_DIR}/fig1_boxplot.png'
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

## Cell 10 — Figure 2: Win-Rate Pie Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Per-Image Win Rate: Method A vs Method B', fontsize=13, fontweight='bold')
for ax, (a_arr, b_arr, metric, direction) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR',  'high'),
    (A_ssim,  B_ssim,  'SSIM',  'high'),
    (A_lpips, B_lpips, 'LPIPS', 'low'),
]):
    wins_B = int((b_arr>a_arr).sum()) if direction=='high' else int((b_arr<a_arr).sum())
    wins_A = len(rows) - wins_B
    ax.pie([wins_A, wins_B],
           labels=[f'A wins ({wins_A})', f'B wins ({wins_B})'],
           colors=[CA, CB], autopct='%1.1f%%', startangle=90,
           textprops={'fontsize': 12},
           wedgeprops=dict(edgecolor='white', linewidth=2))
    ax.set_title(metric, fontsize=13, fontweight='bold')
plt.tight_layout()
p = f'{SAVE_DIR}/fig2_win_rate.png'
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

## Cell 11 — Figure 3: Metric vs Number of Persons

In [ ]:
unique_n = sorted(set(n_pers.astype(int)))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Metric vs Number of Persons per Image', fontsize=13, fontweight='bold')
for ax, (a_arr, b_arr, ylabel) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR (dB)'),
    (A_ssim,  B_ssim,  'SSIM'),
    (A_lpips, B_lpips, 'LPIPS'),
]):
    for n_val in unique_n:
        mask_n = n_pers.astype(int) == n_val
        if not mask_n.any(): continue
        ax.scatter([n_val-0.12]*mask_n.sum(), a_arr[mask_n], c=CA, alpha=0.55, s=30,
                   label='A' if n_val==unique_n[0] else '')
        ax.scatter([n_val+0.12]*mask_n.sum(), b_arr[mask_n], c=CB, alpha=0.55, s=30,
                   label='B' if n_val==unique_n[0] else '')
    valid_n = [n for n in unique_n if (n_pers.astype(int)==n).any()]
    grp_a   = [a_arr[n_pers.astype(int)==n].mean() for n in valid_n]
    grp_b   = [b_arr[n_pers.astype(int)==n].mean() for n in valid_n]
    ax.plot(valid_n, grp_a, 'o--', color='#a00000', lw=2, label='A mean')
    ax.plot(valid_n, grp_b, 's--', color='#1a4f8a', lw=2, label='B mean')
    ax.set_xlabel('# persons'); ax.set_ylabel(ylabel)
    ax.set_xticks(unique_n); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
p = f'{SAVE_DIR}/fig3_metric_vs_persons.png'
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

## Cell 12 — Figure 4: Per-Image Bar Chart

In [ ]:
sort_idx = np.argsort(A_psnr)
x = np.arange(len(rows)); w = 0.4
fig, axes = plt.subplots(3, 1, figsize=(max(22, len(rows)*0.28), 15), sharex=True)
fig.suptitle('Per-Image Metrics — Sorted by Method A PSNR', fontsize=12, fontweight='bold')
for ax, (a_arr, b_arr, ylabel) in zip(axes, [
    (A_psnr[sort_idx],  B_psnr[sort_idx],  'PSNR (dB)'),
    (A_ssim[sort_idx],  B_ssim[sort_idx],  'SSIM'),
    (A_lpips[sort_idx], B_lpips[sort_idx], 'LPIPS'),
]):
    ax.bar(x-w/2, a_arr, w, color=CA, alpha=0.82, label='A: Single-Shot')
    ax.bar(x+w/2, b_arr, w, color=CB, alpha=0.82, label='B: Progressive')
    ax.axhline(a_arr.mean(), color='#900000', lw=1.8, ls='--', alpha=0.8,
               label=f'A mean={a_arr.mean():.3f}')
    ax.axhline(b_arr.mean(), color='#14406b', lw=1.8, ls='--', alpha=0.8,
               label=f'B mean={b_arr.mean():.3f}')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8, loc='upper left'); ax.grid(axis='y', alpha=0.25)
axes[-1].set_xlabel('Image index (sorted by A:PSNR)', fontsize=10)
plt.tight_layout()
p = f'{SAVE_DIR}/fig4_per_image_bars.png'
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

## Cell 13 — Figure 5: Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Correlation Scatter Plots', fontsize=13, fontweight='bold')
axes[0].scatter(A_psnr, A_ssim,  c=CA, alpha=0.6, s=35, label='A: Single-Shot')
axes[0].scatter(B_psnr, B_ssim,  c=CB, alpha=0.6, s=35, label='B: Progressive')
axes[0].set_xlabel('PSNR (dB)'); axes[0].set_ylabel('SSIM')
axes[0].set_title('PSNR vs SSIM'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].scatter(A_psnr, A_lpips, c=CA, alpha=0.6, s=35, label='A: Single-Shot')
axes[1].scatter(B_psnr, B_lpips, c=CB, alpha=0.6, s=35, label='B: Progressive')
axes[1].set_xlabel('PSNR (dB)'); axes[1].set_ylabel('LPIPS')
axes[1].set_title('PSNR vs LPIPS'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
p = f'{SAVE_DIR}/fig5_scatter.png'
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')
print(f'All figures saved to: {SAVE_DIR}')

## Cell 14 — Visual Grid (sample results)

In [ ]:
saved = list(Path(VISUAL_DIR).glob('*_comparison.jpg'))
print(f'Saved visual comparisons: {len(saved)}')

if len(saved) >= 1:
    SHOW   = min(6, len(saved))
    sample = random.sample(saved, SHOW)
    fig, axes = plt.subplots(SHOW, 1, figsize=(28, 7*SHOW))
    if SHOW == 1: axes = [axes]
    fig.suptitle('Sample Visual Results (Original | Detection | Seg Mask | Method A | Method B)',
                 fontsize=13, fontweight='bold')
    for ax, fp in zip(axes, sample):
        ax.imshow(plt.imread(str(fp)))
        ax.set_title(fp.stem, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    pg = f'{SAVE_DIR}/sample_visual_grid.png'
    plt.savefig(pg, dpi=90, bbox_inches='tight'); plt.show()
    print(f'Saved: {pg}')

## Cell 15 — Download All Results as ZIP
সব results (CSV + figures + visuals) একটা zip-এ download করো।

In [ ]:
from google.colab import files

OUT_ZIP = '/content/research_results.zip'
print('Creating results ZIP...')
shutil.make_archive('/content/research_results', 'zip', SAVE_DIR)

size_mb = os.path.getsize(OUT_ZIP) / 1e6
print(f'ZIP size: {size_mb:.1f} MB')
print('Starting download...')
files.download(OUT_ZIP)
print('Download started!')